# Step 4b: Rule Validation

This notebook performs rule validation on extracted document data, validating business rules and compliance requirements.

**Key Features:**
- Policy classification to filter applicable policy types
- Rule validation against extracted attributes
- Orchestration to consolidate results across sections

**Inputs:**
- Document object with extraction results from Step 3
- Rule validation configuration with policy classes

**Outputs:**
- Policy classification results (matched policy types)
- Rule validation results per section
- Consolidated summary across all sections

## 1. Package Installation

In [ ]:
ROOTDIR="../.."

# Let's make sure that modules are autoreloaded
%load_ext autoreload
%autoreload 2

# First uninstall existing package (to ensure we get the latest version)
%pip uninstall -y idp_common

# Install the IDP common package with all components in development mode
%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[dev, all]"

# Check installed version
%pip show idp_common | grep -E "Version|Location"

## 2. Load Previous Step Data

In [ ]:
import os
import json
import time
import logging
import boto3
import yaml
from pathlib import Path

# Import IDP libraries
from idp_common.models import Document, Status, RuleValidationResult
from idp_common.rule_validation import (
    PolicyClassificationService,
    RuleValidationService,
    RuleValidationOrchestratorService
)

# Configure logging
logging.basicConfig(level=logging.WARNING)
logging.getLogger('idp_common.rule_validation').setLevel(logging.INFO)
logging.getLogger('idp_common.bedrock.client').setLevel(logging.INFO)

print("Libraries imported successfully")

In [ ]:
# Load document from previous step (assessment)
assessment_data_dir = Path(".data/step4_assessment")

# Check if data exists
if not assessment_data_dir.exists():
    print("⚠️ No data from previous step found.")
    print("Please run the previous notebooks first:")
    print("  1. step0_setup.ipynb")
    print("  2. step1_ocr.ipynb")
    print("  3. step2_classification.ipynb")
    print("  4. step3_extraction.ipynb")
    print("  5. step4_assessment.ipynb")
    raise FileNotFoundError(f"Directory {assessment_data_dir} not found")

# Load document object from JSON
document_path = assessment_data_dir / "document.json"
with open(document_path, 'r') as f:
    document = Document.from_json(f.read())

# Load rule validation configuration
config_path = Path("config/rule_validation.yaml")
if config_path.exists():
    with open(config_path, 'r') as f:
        CONFIG = yaml.safe_load(f)
    print(f"Loaded rule validation config from {config_path}")
else:
    # Fallback to config_library
    config_path = Path(f"{ROOTDIR}/config_library/unified/rule-validation/config.yaml")
    with open(config_path, 'r') as f:
        CONFIG = yaml.safe_load(f)
    print(f"Loaded rule validation config from {config_path}")

# Load environment info
env_path = assessment_data_dir / "environment.json"
with open(env_path, 'r') as f:
    env_info = json.load(f)

# Set environment variables
region = env_info['region']
os.environ['AWS_REGION'] = region
os.environ['METRIC_NAMESPACE'] = 'IDP-Modular-Pipeline'

print(f"\nLoaded document: {document.id}")
print(f"Document status: {document.status.value}")
print(f"Number of sections: {len(document.sections) if document.sections else 0}")
print(f"Region: {region}")

## 3. Policy Classification

Policy classification determines which policy types apply to this document based on:
- Document name regex matching
- Page content regex matching

If no policy types match, rule validation will be skipped.

In [ ]:
# Create policy classification service
policy_classification_service = PolicyClassificationService(config=CONFIG)

print("Policy Classification Service initialized")
print(f"Policy classes configured: {policy_classification_service.get_all_policy_types()}")

In [ ]:
# Classify the document to determine which policy classes apply
print("Classifying document for applicable policy classes...")
start_time = time.time()

classification_result = policy_classification_service.classify_document(document)

policy_classification_time = time.time() - start_time
print(f"Policy classification completed in {policy_classification_time:.2f} seconds")

# ALWAYS set document.rule_validation_result
document.rule_validation_result = RuleValidationResult(
    request_id=document.id,
    matched_policy_types=classification_result.matched_policy_types,
    matched_page_ids=classification_result.matched_page_ids
)

# Set skip flag based on results
if classification_result.matched_policy_types:
    print(f"\n✅ Matched policy types: {classification_result.matched_policy_types}")
    if classification_result.matched_page_ids:
        print(f"   Matched page IDs by policy type: {classification_result.matched_page_ids}")
    skip_rule_validation = False
else:
    print(f"\n⚠️ No policy classes matched - rule validation will be skipped")
    print(f"   No regex patterns matched the document name or page content")
    skip_rule_validation = True

print(f"\nDocument updated: matched_policy_types={document.rule_validation_result.matched_policy_types}")

## 4. Rule Validation

Process each document section through rule validation to check compliance with the matched policy types.

In [ ]:
if skip_rule_validation:
    print("⏭️ Skipping rule validation - no policy classes matched")
    section_results = []
else:
    # Process each section individually
    section_results = []
    
    sections_to_process = document.sections if document.sections else []
    print(f"Processing {len(sections_to_process)} section(s)...")
    
    for section in sections_to_process:
        print(f"\n--- Processing section {section.section_id} ({section.classification}) ---")
        
        # Create a document with only this section
        section_document = Document(
            id=document.id,
            input_key=document.input_key,
            input_bucket=document.input_bucket,
            output_bucket=document.output_bucket,
            pages=document.pages,
            sections=[section],
            status=document.status,
            metering=document.metering.copy() if document.metering else {},
            rule_validation_result=document.rule_validation_result,
        )
        
        # Create fresh service instance for each section
        section_rule_validation_service = RuleValidationService(
            region=region,
            config=CONFIG
        )
        
        # Process the single section
        start_time = time.time()
        section_result = section_rule_validation_service.validate_document(section_document)
        processing_time = time.time() - start_time
        
        section_results.append(section_result)
        print(f"✅ Completed section {section.section_id} in {processing_time:.2f}s")
    
    print(f"\n🎉 Rule validation complete for {len(section_results)} section(s)")

## 5. Display Section Results

In [ ]:
if skip_rule_validation:
    print("⏭️ Rule validation was skipped - no results to display")
else:
    print("Section Results Summary:")
    print("=" * 50)
    
    for i, section_result in enumerate(section_results):
        section_id = document.sections[i].section_id
        if hasattr(section_result, 'rule_validation_result') and section_result.rule_validation_result:
            rv_result = section_result.rule_validation_result
            section_uri = rv_result.metadata.get('section_output_uri') if rv_result.metadata else None
            if section_uri:
                print(f"\nSection {section_id}:")
                print(f"  Results saved to: {section_uri}")
            else:
                print(f"\nSection {section_id}: Rule validation completed")
        else:
            print(f"\nSection {section_id}: No rule validation results")

## 6. Orchestration (Consolidation)

Consolidate results from all sections into a unified summary.

In [ ]:
if skip_rule_validation:
    print("⏭️ Orchestration skipped - no policy classes matched")
else:
    # Create Orchestration service
    orchestration_service = RuleValidationOrchestratorService(config=CONFIG)
    print("Orchestration service created")
    
    # Run consolidation
    print("\nConsolidating results across sections...")
    start_time = time.time()
    
    consolidated_document = orchestration_service.consolidate_and_save(document, CONFIG)
    
    consolidation_time = time.time() - start_time
    print(f"Consolidation completed in {consolidation_time:.2f} seconds")
    
    # Display consolidated results
    if consolidated_document.rule_validation_result:
        rv_result = consolidated_document.rule_validation_result
        print(f"\n=== Consolidated Results ===")
        if rv_result.output_uri:
            print(f"Summary URI: {rv_result.output_uri}")
        if rv_result.summary:
            print(f"Policy type URIs: {len(rv_result.summary.get('policy_type_uris', []))}")
            if rv_result.summary.get('consolidated_summary_uri'):
                print(f"Consolidated summary: {rv_result.summary['consolidated_summary_uri']}")

## 7. Save Results for Next Step

In [ ]:
# Save document state for next step
output_dir = Path(".data/step5_rule_validation")
output_dir.mkdir(parents=True, exist_ok=True)

# Save document object
document_to_save = consolidated_document if not skip_rule_validation else document
document_path = output_dir / "document.json"
with open(document_path, 'w') as f:
    f.write(document_to_save.to_json())

# Download rule validation results from S3
rv_result = document_to_save.rule_validation_result
consolidated_data = None
policy_type_results = {}

if rv_result and rv_result.summary:
    from idp_common import s3
    
    # Download consolidated summary
    if rv_result.summary.get('consolidated_summary_uri'):
        try:
            consolidated_uri = rv_result.summary['consolidated_summary_uri']
            json_uri = consolidated_uri.replace('.md', '.json') if consolidated_uri.endswith('.md') else consolidated_uri
            consolidated_data = json.loads(s3.get_text_content(json_uri))
        except:
            pass
    
    # Download policy type results
    for uri in rv_result.summary.get('policy_type_uris', []):
        try:
            policy_type = uri.split('/')[-1].replace('_responses.json', '')
            policy_type_results[policy_type] = json.loads(s3.get_text_content(uri))
        except:
            pass

# Build rule validation summary
summary = {
    'document_id': document_to_save.id,
    'skip_rule_validation': skip_rule_validation,
    'policy_classification': {
        'matched_policy_types': classification_result.matched_policy_types,
        'matched_page_ids': classification_result.matched_page_ids
    },
    'rule_validation': {
        'sections_processed': len(section_results),
        'output_uri': rv_result.output_uri if rv_result else None,
        'policy_type_uris': rv_result.summary.get('policy_type_uris', []) if rv_result and rv_result.summary else [],
        'consolidated_summary_uri': rv_result.summary.get('consolidated_summary_uri') if rv_result and rv_result.summary else None,
        'consolidated_results': consolidated_data,
        'policy_type_results': policy_type_results
    }
}
summary_path = output_dir / "rule_validation_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Save configuration
config_path = output_dir / "config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

# Save environment info
env_path = output_dir / "environment.json"
with open(env_path, 'w') as f:
    json.dump(env_info, f, indent=2)

print(f"✅ Results saved to {output_dir}")
print(f"   - document.json")
print(f"   - rule_validation_summary.json")
print(f"   - config.json")
print(f"   - environment.json")

## Summary

This notebook demonstrated the rule validation workflow:

1. **Policy Classification** - Determined which policy types apply based on document name and content regex matching
2. **Rule Validation** - Processed each section against matched policy types
3. **Orchestration** - Consolidated results across all sections

**Next Steps:**
- Proceed to Step 6 (Summarization) for document summarization
- Review the consolidated summary in S3 for detailed rule validation results